# spaceapps-core — Quickstart for the ML team

This notebook shows how to pull data from each source **without starting the HTTP server**.
Every client returns the same `TimeSeries` schema and has a `.to_dataframe()` helper, so the data is immediately usable with pandas / xgboost / scikit-learn.

Pilot zone used in examples: Piura, Peru (lat = -5.19, lon = -80.63).

In [ ]:
from datetime import date

from spaceapps_core.clients.power import PowerClient
from spaceapps_core.clients.open_meteo import OpenMeteoClient
from spaceapps_core.schemas.common import Location

LOCATION = Location(lat=-5.19, lon=-80.63)

## 1. NASA POWER — daily history

No credentials required.

In [ ]:
series = await PowerClient().daily_point(
    location=LOCATION,
    start=date(2024, 1, 1),
    end=date(2024, 1, 31),
)

df = PowerClient.to_dataframe(series)
df.head()

## 2. Open-Meteo — forecast

No credentials required. 1–16 days ahead.

In [ ]:
forecast = await OpenMeteoClient().forecast(location=LOCATION, forecast_days=10)
OpenMeteoClient.to_dataframe(forecast).head()

## 3. NEX-GDDP-CMIP6 — projections

Anonymous S3 (no credentials). Slow — expect several seconds per year.

In [ ]:
from spaceapps_core.clients.nex_gddp import NexGddpClient
from spaceapps_core.schemas.variables import CanonicalVariable

proj = await NexGddpClient().projections_point(
    location=LOCATION,
    start=date(2050, 1, 1),
    end=date(2050, 1, 31),
    scenario='ssp245',
    variables=[CanonicalVariable.T2M_MAX, CanonicalVariable.PRECIPITATION],
)
NexGddpClient.to_dataframe(proj).head()

## 4. Multi-source feature table

Build a wide pandas DataFrame indexed by date, with one column per canonical variable, ready for ML training.

In [ ]:
from spaceapps_core.services.features import build_features

features = await build_features(
    location=LOCATION,
    start=date(2024, 1, 1),
    end=date(2024, 1, 31),
    include_forecast=False,
    include_imerg=False,  # set True when Earthdata is configured
)
features.head()

## 5. AppEEARS — asynchronous jobs (NDVI/EVI, SMAP)

Requires `EARTHDATA_USERNAME` and `EARTHDATA_PASSWORD` in `.env`.
The workflow is: submit a task, poll until done, then parse the point CSV result.

In [ ]:
from spaceapps_core.clients.appeears import AppEEARSClient, MODIS_NDVI_LAYERS

appeears = AppEEARSClient()
# task_id = await appeears.submit_point_task(
#     location=LOCATION,
#     start=date(2024, 1, 1),
#     end=date(2024, 2, 29),
#     layers=MODIS_NDVI_LAYERS,
# )
# task_id